# 📊 Tutorial de Plotly para Visualización Científica
## Gráficas 2D, 3D y Animaciones con Aplicaciones en Física y Astronomía

**Universidad de Antioquia — Modelamiento N Cuerpos**

---

Este cuaderno es una clase práctica sobre la librería **Plotly** de Python, orientada a su uso
en simulaciones de sistemas físicos y astronómicos. Al finalizar, serás capaz de:

- Crear gráficas 2D interactivas (líneas, dispersión, mapas de colores, subplots).
- Crear gráficas 3D interactivas (curvas, superficies, nubes de puntos).
- Producir animaciones de sistemas dinámicos describibles por EDOs.

### Contenido

| Sección | Tema |
|---------|------|
| 0 | Instalación e importaciones |
| 1 | Plotly 2D — Fundamentos |
| 2 | Plotly 2D — Sistemas de EDOs en física |
| 3 | Plotly 3D — Curvas, superficies y atractores |
| 4 | Animaciones con Plotly |

> **Nota:** todas las celdas son ejecutables; los gráficos son interactivos (zoom, rotación, exportación).


## Sección 0 — Instalación e Importaciones

In [ ]:
# Instalar dependencias (descomentar si es necesario)
# !pip install plotly numpy scipy kaleido


In [ ]:
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from scipy.integrate import solve_ivp

# Configuración estética global
TEMPLATE = "plotly_dark"   # cambia a "plotly", "ggplot2", "seaborn", etc.
print("✅ Librerías importadas correctamente")
print(f"   Plotly versión: {go.__module__.split('.')[0]}")


### Estructura básica de una figura Plotly

Plotly organiza sus gráficas en tres capas:

```
go.Figure(
    data  = [traces],   # ← qué se grafica (líneas, puntos, superficies…)
    layout = go.Layout( # ← cómo se ve (título, ejes, fondo…)
        title = "Mi gráfica",
        xaxis_title = "x",
        yaxis_title = "y"
    )
)
```

Los **traces** más usados son:

| Trace | Descripción |
|-------|-------------|
| `go.Scatter` | Líneas y puntos 2D |
| `go.Scatter3d` | Líneas y puntos 3D |
| `go.Surface` | Superficies 3D |
| `go.Heatmap` | Mapas de calor |
| `go.Cone` | Campos vectoriales 3D |


---
## Sección 1 — Plotly 2D: Fundamentos

### 1.1 Primer gráfico: funciones matemáticas

In [ ]:
x = np.linspace(0, 4 * np.pi, 500)

fig = go.Figure()

fig.add_trace(go.Scatter(x=x, y=np.sin(x),
                         mode="lines", name="sin(x)",
                         line=dict(color="cyan", width=2)))

fig.add_trace(go.Scatter(x=x, y=np.cos(x),
                         mode="lines", name="cos(x)",
                         line=dict(color="orange", width=2, dash="dash")))

fig.add_trace(go.Scatter(x=x, y=np.sin(x) * np.exp(-x / 5),
                         mode="lines", name="sin(x)·e^{-x/5}",
                         line=dict(color="lightgreen", width=2)))

fig.update_layout(
    template=TEMPLATE,
    title="Funciones trigonométricas",
    xaxis_title="x (rad)",
    yaxis_title="Amplitud",
    legend=dict(x=0.8, y=1),
    width=900, height=450
)

fig.show()


### 1.2 Modos de visualización: `lines`, `markers`, `lines+markers`

In [ ]:
t = np.linspace(0, 2 * np.pi, 20)

fig = make_subplots(rows=1, cols=3,
                    subplot_titles=["mode='lines'",
                                    "mode='markers'",
                                    "mode='lines+markers'"])

for col, mode in enumerate(["lines", "markers", "lines+markers"], start=1):
    fig.add_trace(
        go.Scatter(x=t, y=np.sin(t), mode=mode,
                   line=dict(color="deepskyblue"),
                   marker=dict(color="tomato", size=8),
                   showlegend=False),
        row=1, col=col
    )

fig.update_layout(template=TEMPLATE, title="Modos de trazado en Plotly",
                  width=900, height=350)
fig.show()


### 1.3 Personalización: colores, anotaciones y zonas sombreadas

In [ ]:
x = np.linspace(0, 10, 400)
y = np.exp(-0.3 * x) * np.cos(2 * np.pi * x)

fig = go.Figure()

# Relleno hacia el eje x
fig.add_trace(go.Scatter(
    x=x, y=y,
    fill="tozeroy", fillcolor="rgba(0,180,255,0.15)",
    line=dict(color="deepskyblue", width=2),
    name="x(t)"
))

# Línea y = 0
fig.add_hline(y=0, line_dash="dot", line_color="white", opacity=0.4)

# Anotación de un punto especial
max_idx = np.argmax(y)
fig.add_annotation(
    x=x[max_idx], y=y[max_idx],
    text="Máximo",
    showarrow=True, arrowhead=2,
    font=dict(color="yellow", size=13),
    arrowcolor="yellow"
)

# Zona sombreada: región de tiempo
fig.add_vrect(x0=5, x1=7,
              fillcolor="rgba(255,50,50,0.1)",
              layer="below", line_width=0,
              annotation_text="Región crítica",
              annotation_position="top left",
              annotation=dict(font_color="salmon"))

fig.update_layout(
    template=TEMPLATE,
    title="Oscilador amortiguado — personalización avanzada",
    xaxis_title="Tiempo (s)",
    yaxis_title="Desplazamiento",
    width=900, height=420
)
fig.show()


### 1.4 Subplots y ejes secundarios

In [ ]:
from plotly.subplots import make_subplots

t = np.linspace(0, 20, 1000)
x = np.exp(-0.1 * t) * np.cos(t)
v = np.exp(-0.1 * t) * (-0.1 * np.cos(t) - np.sin(t))

fig = make_subplots(
    rows=2, cols=2,
    specs=[[{"colspan": 2}, None],
           [{}, {}]],
    subplot_titles=["Posición x(t)",
                    "Velocidad v(t)", "Espacio de fase (x, v)"]
)

# Fila 1: posición
fig.add_trace(go.Scatter(x=t, y=x, mode="lines",
                         line=dict(color="cyan"), name="x(t)"),
              row=1, col=1)

# Fila 2 izq: velocidad
fig.add_trace(go.Scatter(x=t, y=v, mode="lines",
                         line=dict(color="orange"), name="v(t)"),
              row=2, col=1)

# Fila 2 der: espacio de fase
fig.add_trace(go.Scatter(x=x, y=v, mode="lines",
                         line=dict(color="lightgreen", width=1),
                         name="Espacio de fase"),
              row=2, col=2)

fig.update_xaxes(title_text="Tiempo (s)", row=1, col=1)
fig.update_xaxes(title_text="Tiempo (s)", row=2, col=1)
fig.update_xaxes(title_text="x", row=2, col=2)
fig.update_yaxes(title_text="x(t)", row=1, col=1)
fig.update_yaxes(title_text="v(t)", row=2, col=1)
fig.update_yaxes(title_text="v(t)", row=2, col=2)

fig.update_layout(template=TEMPLATE, title="Dashboard: oscilador amortiguado",
                  height=600, width=900, showlegend=False)
fig.show()


### 1.5 Gráficas de dispersión con color como cuarta variable

In [ ]:
# Simulación de partículas con energía como color
rng = np.random.default_rng(42)
N = 300
theta = rng.uniform(0, 2 * np.pi, N)
r = rng.rayleigh(2, N)
x_p = r * np.cos(theta)
y_p = r * np.sin(theta)
energia = 0.5 * r**2  # E = ½mv² proporcional

fig = go.Figure(go.Scatter(
    x=x_p, y=y_p,
    mode="markers",
    marker=dict(
        color=energia,
        colorscale="Viridis",
        size=7,
        colorbar=dict(title="Energía cinética"),
        showscale=True,
        opacity=0.8
    ),
    text=[f"E = {e:.2f}" for e in energia],
    hoverinfo="text"
))

fig.update_layout(
    template=TEMPLATE,
    title="Distribución de partículas coloreadas por energía cinética",
    xaxis_title="x (u.a.)",
    yaxis_title="y (u.a.)",
    width=700, height=550,
    yaxis=dict(scaleanchor="x")  # mantener proporción 1:1
)
fig.show()


---
## Sección 2 — Sistemas de EDOs en Física con Plotly 2D

En física muchos sistemas se describen mediante **Ecuaciones Diferenciales Ordinarias (EDOs)**.
Las integraremos numéricamente con `scipy.integrate.solve_ivp` (método RK45 por defecto)
y visualizaremos los resultados con Plotly.


### 2.1 Péndulo Simple

La ecuación de movimiento del péndulo simple de longitud $L$ es:

$$\ddot{\theta} + \frac{g}{L}\sin\theta = 0$$

Reducimos a sistema de primer orden:

$$\begin{cases}
\dot{\theta} = \omega \\
\dot{\omega} = -\dfrac{g}{L}\sin\theta
\end{cases}$$

Compararemos soluciones para ángulos pequeños (lineal) y grandes (no lineal).


In [ ]:
g = 9.81   # m/s²
L = 1.0    # m

def pendulo(t, y):
    theta, omega = y
    return [omega, -(g / L) * np.sin(theta)]

def pendulo_lineal(t, y):
    theta, omega = y
    return [omega, -(g / L) * theta]

t_span = (0, 10)
t_eval = np.linspace(*t_span, 2000)

angulos = [5, 30, 60, 90, 150]  # grados

fig = go.Figure()

colors = px.colors.sequential.Plasma_r
for i, ang_deg in enumerate(angulos):
    theta0 = np.radians(ang_deg)
    sol = solve_ivp(pendulo, t_span, [theta0, 0], t_eval=t_eval, rtol=1e-8)
    fig.add_trace(go.Scatter(
        x=sol.t, y=np.degrees(sol.y[0]),
        mode="lines",
        name=f"θ₀ = {ang_deg}°",
        line=dict(color=colors[i], width=2)
    ))

# Solución lineal para θ₀ = 5°
sol_lin = solve_ivp(pendulo_lineal, t_span, [np.radians(5), 0],
                    t_eval=t_eval, rtol=1e-8)
fig.add_trace(go.Scatter(
    x=sol_lin.t, y=np.degrees(sol_lin.y[0]),
    mode="lines", name="Lineal (5°)",
    line=dict(color="white", width=1.5, dash="dash")
))

fig.update_layout(
    template=TEMPLATE,
    title="Péndulo simple — comparación lineal vs. no lineal",
    xaxis_title="Tiempo (s)",
    yaxis_title="Ángulo θ (°)",
    width=900, height=450,
    legend=dict(x=0.85, y=1)
)
fig.show()


#### Espacio de fase del péndulo

In [ ]:
fig = go.Figure()

for i, ang_deg in enumerate(angulos):
    theta0 = np.radians(ang_deg)
    sol = solve_ivp(pendulo, t_span, [theta0, 0], t_eval=t_eval, rtol=1e-8)
    fig.add_trace(go.Scatter(
        x=np.degrees(sol.y[0]),
        y=np.degrees(sol.y[1]),
        mode="lines",
        name=f"θ₀ = {ang_deg}°",
        line=dict(color=colors[i], width=2)
    ))

fig.update_layout(
    template=TEMPLATE,
    title="Espacio de fase del péndulo simple",
    xaxis_title="θ (°)",
    yaxis_title="ω (°/s)",
    width=700, height=500
)
fig.show()


### 2.2 Oscilador Armónico Amortiguado Forzado

$$\ddot{x} + 2\gamma\dot{x} + \omega_0^2 x = \frac{F_0}{m}\cos(\Omega t)$$

Sistema de primer orden:

$$\begin{cases}
\dot{x} = v \\
\dot{v} = -2\gamma v - \omega_0^2 x + \frac{F_0}{m}\cos(\Omega t)
\end{cases}$$

Estudiaremos la **resonancia** variando la frecuencia de forzamiento $\Omega$.


In [ ]:
omega0 = 2.0   # frecuencia natural (rad/s)
gamma = 0.1    # coeficiente de amortiguamiento
F0 = 1.0       # amplitud de fuerza
m = 1.0

def oscilador_forzado(t, y, Omega):
    x, v = y
    return [v, -2*gamma*v - omega0**2 * x + (F0/m)*np.cos(Omega*t)]

t_span_f = (0, 80)
t_eval_f = np.linspace(*t_span_f, 8000)

omegas = [1.0, 1.5, 1.9, 2.0, 2.1, 2.5, 3.0]  # cerca de la resonancia

fig = go.Figure()
colorscale = px.colors.diverging.RdYlGn

for i, Om in enumerate(omegas):
    sol = solve_ivp(oscilador_forzado, t_span_f, [0, 0],
                    t_eval=t_eval_f, args=(Om,), rtol=1e-9, max_step=0.01)
    # amplitud en estado estacionario (últimos 20%)
    idx_ss = int(0.8 * len(sol.t))
    amp = np.max(np.abs(sol.y[0][idx_ss:]))
    
    fig.add_trace(go.Scatter(
        x=sol.t, y=sol.y[0],
        mode="lines",
        name=f"Ω = {Om:.1f} rad/s  (A≈{amp:.2f})",
        line=dict(width=1.5),
        visible=(i == 3)  # solo muestra resonancia por defecto
    ))

# Botones para seleccionar frecuencia
buttons = []
for i, Om in enumerate(omegas):
    visible = [j == i for j in range(len(omegas))]
    buttons.append(dict(
        label=f"Ω = {Om:.1f}",
        method="update",
        args=[{"visible": visible}]
    ))

fig.update_layout(
    template=TEMPLATE,
    title="Oscilador forzado — explorar resonancia",
    xaxis_title="Tiempo (s)",
    yaxis_title="x(t)",
    updatemenus=[dict(
        buttons=buttons,
        direction="down",
        x=0.12, y=1.15
    )],
    width=900, height=450
)
fig.show()


#### Curva de resonancia — amplitud vs. frecuencia forzada

In [ ]:
Omega_arr = np.linspace(0.5, 4.0, 300)
amplitudes = []

t_long = (0, 200)
t_ev = np.linspace(*t_long, 20000)

for Om in Omega_arr:
    sol = solve_ivp(oscilador_forzado, t_long, [0, 0],
                    t_eval=t_ev, args=(Om,), rtol=1e-8, max_step=0.05)
    idx_ss = int(0.85 * len(sol.t))
    amplitudes.append(np.max(np.abs(sol.y[0][idx_ss:])))

# Curva analítica
amp_analitica = (F0/m) / np.sqrt((omega0**2 - Omega_arr**2)**2 + (2*gamma*Omega_arr)**2)

fig = go.Figure()
fig.add_trace(go.Scatter(x=Omega_arr, y=amplitudes,
                         mode="lines", name="Numérica",
                         line=dict(color="cyan", width=2)))
fig.add_trace(go.Scatter(x=Omega_arr, y=amp_analitica,
                         mode="lines", name="Analítica",
                         line=dict(color="orange", width=2, dash="dash")))
fig.add_vline(x=omega0, line_dash="dot", line_color="red",
              annotation_text=f"ω₀ = {omega0}", annotation_position="top right")

fig.update_layout(
    template=TEMPLATE,
    title=f"Curva de resonancia (γ = {gamma})",
    xaxis_title="Ω (rad/s)",
    yaxis_title="Amplitud estacionaria",
    width=800, height=420
)
fig.show()


### 2.3 Sistema de Lotka-Volterra (Depredador-Presa)

Las ecuaciones de Lotka-Volterra modelan la dinámica de dos especies:

$$\begin{cases}
\dot{x} = \alpha x - \beta x y \\
\dot{y} = \delta x y - \gamma y
\end{cases}$$

donde $x$ = presas, $y$ = depredadores.


In [ ]:
alpha = 1.0   # tasa de crecimiento de presas
beta  = 0.1   # tasa de depredación
delta = 0.075 # eficiencia de conversión
gamma = 1.5   # tasa de muerte de depredadores

def lotka_volterra(t, y):
    x, predators = y
    return [alpha*x - beta*x*predators,
            delta*x*predators - gamma*predators]

t_span_lv = (0, 100)
t_eval_lv = np.linspace(*t_span_lv, 5000)

sol = solve_ivp(lotka_volterra, t_span_lv, [10, 5],
                t_eval=t_eval_lv, rtol=1e-8)

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=["Dinámica temporal", "Espacio de fase"])

fig.add_trace(go.Scatter(x=sol.t, y=sol.y[0],
                         mode="lines", name="Presas (x)",
                         line=dict(color="limegreen", width=2)),
              row=1, col=1)
fig.add_trace(go.Scatter(x=sol.t, y=sol.y[1],
                         mode="lines", name="Depredadores (y)",
                         line=dict(color="tomato", width=2)),
              row=1, col=1)

# Espacio de fase coloreado por tiempo
fig.add_trace(go.Scatter(
    x=sol.y[0], y=sol.y[1],
    mode="lines",
    line=dict(color=sol.t, colorscale="Plasma", width=2),
    showlegend=False,
    hovertemplate="Presas: %{x:.1f}<br>Depred.: %{y:.1f}<extra></extra>"
), row=1, col=2)

fig.update_xaxes(title_text="Tiempo", row=1, col=1)
fig.update_xaxes(title_text="Presas (x)", row=1, col=2)
fig.update_yaxes(title_text="Población", row=1, col=1)
fig.update_yaxes(title_text="Depredadores (y)", row=1, col=2)

fig.update_layout(template=TEMPLATE,
                  title="Sistema de Lotka-Volterra",
                  width=1000, height=420)
fig.show()


### 2.4 Campo vectorial y trayectorias — retratos de fase 2D

Visualizaremos el **campo vectorial** de un sistema y varias trayectorias superpuestas
usando un mapa de calor para la magnitud del campo.


In [ ]:
# Campo vectorial del péndulo simple (sistema autónomo)
theta_g = np.linspace(-np.pi, np.pi, 25)
omega_g = np.linspace(-4, 4, 25)
TH, OM = np.meshgrid(theta_g, omega_g)

dTH = OM
dOM = -(g / L) * np.sin(TH)
mag = np.sqrt(dTH**2 + dOM**2) + 1e-12

# Normalizar
U = dTH / mag
V = dOM / mag

fig = go.Figure()

# Campo vectorial como flechas (usando anotaciones)
# Representación simplificada: puntos con tamaño proporcional a |F|
mag_norm = mag / mag.max()
fig.add_trace(go.Scatter(
    x=TH.ravel(), y=OM.ravel(),
    mode="markers",
    marker=dict(
        color=mag.ravel(),
        colorscale="Magma",
        size=6,
        colorbar=dict(title="|F|"),
        showscale=True
    ),
    showlegend=False,
    hoverinfo="skip"
))

# Trayectorias para distintas condiciones iniciales
for theta_init in [-2.5, -np.pi*0.8, 0.5, np.pi*0.9]:
    for omega_init in [-2, 0, 2]:
        sol_ph = solve_ivp(pendulo, (0, 15), [theta_init, omega_init],
                           t_eval=np.linspace(0, 15, 1000), rtol=1e-9)
        fig.add_trace(go.Scatter(
            x=sol_ph.y[0], y=sol_ph.y[1],
            mode="lines",
            line=dict(width=1.5, color="cyan"),
            showlegend=False,
            opacity=0.7
        ))

fig.update_layout(
    template=TEMPLATE,
    title="Retrato de fase del péndulo simple",
    xaxis_title="θ (rad)",
    yaxis_title="ω (rad/s)",
    xaxis=dict(range=[-np.pi, np.pi]),
    yaxis=dict(range=[-4.2, 4.2]),
    width=800, height=500
)
fig.show()


### 2.5 Dos cuerpos — Órbitas de Kepler en 2D

La interacción gravitacional entre dos cuerpos de masas $m_1$ y $m_2$ se reduce al
problema de **un cuerpo** con masa reducida $\mu = m_1 m_2/(m_1+m_2)$:

$$\ddot{\vec{r}} = -\frac{G M}{r^3}\vec{r}, \quad M = m_1 + m_2$$

Convertido a primer orden ($\vec{q} = [x, y, v_x, v_y]$):

$$\dot{q}_1 = q_3,\quad \dot{q}_2 = q_4,\quad
\dot{q}_3 = -\frac{GM x}{r^3},\quad \dot{q}_4 = -\frac{GM y}{r^3}$$


In [ ]:
G = 6.674e-11   # SI
M_sol = 1.989e30  # kg

# Unidades: UA, años, Masas Solares
G_au = 4 * np.pi**2   # G en (UA³ / (Msol · año²))

def kepler_2d(t, y, GM=G_au):
    x, px, vx, vy = y
    # reutilizamos: x,y  = y[0],y[1]; vx,vy = y[2],y[3]
    xi, yi, vxi, vyi = y
    r = np.sqrt(xi**2 + yi**2)
    ax = -GM * xi / r**3
    ay = -GM * yi / r**3
    return [vxi, vyi, ax, ay]

# Condiciones iniciales: Tierra (aprox)
a_tierra = 1.0       # UA (semieje mayor)
e_tierra = 0.0167    # excentricidad

# v en perihelio
v_p = np.sqrt(G_au * (1 + e_tierra) / (a_tierra * (1 - e_tierra)))

y0_tierra = [a_tierra * (1 - e_tierra), 0, 0, v_p]

# Marte
a_marte = 1.524
e_marte = 0.093
v_p_m = np.sqrt(G_au * (1 + e_marte) / (a_marte * (1 - e_marte)))
y0_marte = [a_marte * (1 - e_marte), 0, 0, v_p_m]

# Mercurio
a_merc = 0.387
e_merc = 0.205
v_p_merc = np.sqrt(G_au * (1 + e_merc) / (a_merc * (1 - e_merc)))
y0_merc = [a_merc * (1 - e_merc), 0, 0, v_p_merc]

t_span_orb = (0, 2)   # 2 años
t_eval_orb = np.linspace(*t_span_orb, 3000)

planetas = [
    ("Mercurio", y0_merc, "gold"),
    ("Tierra",   y0_tierra, "deepskyblue"),
    ("Marte",    y0_marte, "tomato"),
]

fig = go.Figure()

for nombre, y0, color in planetas:
    sol = solve_ivp(kepler_2d, t_span_orb, y0, t_eval=t_eval_orb, rtol=1e-10)
    fig.add_trace(go.Scatter(
        x=sol.y[0], y=sol.y[1],
        mode="lines",
        name=nombre,
        line=dict(color=color, width=2)
    ))
    # Punto del planeta en t final
    fig.add_trace(go.Scatter(
        x=[sol.y[0][-1]], y=[sol.y[1][-1]],
        mode="markers",
        marker=dict(color=color, size=12),
        showlegend=False
    ))

# Sol
fig.add_trace(go.Scatter(
    x=[0], y=[0],
    mode="markers",
    marker=dict(color="yellow", size=20, symbol="circle"),
    name="Sol"
))

fig.update_layout(
    template=TEMPLATE,
    title="Órbitas de Kepler — Sistema Solar interior (2 años)",
    xaxis_title="x (UA)",
    yaxis_title="y (UA)",
    yaxis=dict(scaleanchor="x"),
    width=700, height=600
)
fig.show()


---
## Sección 3 — Plotly 3D: Curvas, Superficies y Atractores

Plotly permite rotar e inspeccionar gráficas 3D directamente en el navegador.


### 3.1 Hélice y curvas 3D básicas

In [ ]:
t3d = np.linspace(0, 6 * np.pi, 800)

fig = go.Figure()

fig.add_trace(go.Scatter3d(
    x=np.cos(t3d),
    y=np.sin(t3d),
    z=t3d / (6 * np.pi),
    mode="lines",
    line=dict(color=t3d, colorscale="Rainbow", width=4),
    name="Hélice"
))

fig.update_layout(
    template=TEMPLATE,
    title="Hélice paramétrica 3D",
    scene=dict(
        xaxis_title="x",
        yaxis_title="y",
        zaxis_title="z",
        bgcolor="rgb(5,5,20)"
    ),
    width=700, height=600
)
fig.show()


### 3.2 Atractor de Lorenz

El sistema de Lorenz es un modelo simplificado de convección atmosférica:

$$\begin{cases}
\dot{x} = \sigma(y - x) \\
\dot{y} = x(\rho - z) - y \\
\dot{z} = xy - \beta z
\end{cases}$$

Con $\sigma = 10,\; \rho = 28,\; \beta = 8/3$ el sistema exhibe **caos determinístico**.


In [ ]:
sigma, rho, beta = 10.0, 28.0, 8/3

def lorenz(t, y):
    x, y_c, z = y
    return [sigma * (y_c - x),
            x * (rho - z) - y_c,
            x * y_c - beta * z]

t_span_lor = (0, 40)
t_eval_lor = np.linspace(*t_span_lor, 50000)

y0_lor = [1.0, 1.0, 1.0]
sol_lor = solve_ivp(lorenz, t_span_lor, y0_lor, t_eval=t_eval_lor,
                    method="RK45", rtol=1e-10, atol=1e-12)

x_l, y_c_l, z_l = sol_lor.y

# Color por tiempo
fig = go.Figure(go.Scatter3d(
    x=x_l, y=y_c_l, z=z_l,
    mode="lines",
    line=dict(
        color=t_eval_lor,
        colorscale="Inferno",
        width=2
    ),
    hoverinfo="skip"
))

fig.update_layout(
    template=TEMPLATE,
    title="Atractor de Lorenz (σ=10, ρ=28, β=8/3)",
    scene=dict(
        xaxis_title="x",
        yaxis_title="y",
        zaxis_title="z",
        bgcolor="rgb(5,5,20)",
        camera=dict(eye=dict(x=1.5, y=1.5, z=1.0))
    ),
    width=850, height=650
)
fig.show()


#### Sensibilidad a condiciones iniciales — efecto mariposa

In [ ]:
epsilon = 1e-5
y0_pert = [1.0 + epsilon, 1.0, 1.0]

sol_pert = solve_ivp(lorenz, t_span_lor, y0_pert, t_eval=t_eval_lor,
                     method="RK45", rtol=1e-10, atol=1e-12)

diferencia = np.sqrt(np.sum((sol_lor.y - sol_pert.y)**2, axis=0))

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=t_eval_lor, y=np.log10(diferencia + 1e-20),
    mode="lines",
    line=dict(color="orangered", width=2),
    name="log₁₀ |δ(t)|"
))
fig.add_annotation(
    x=15, y=0,
    text=f"ε₀ = {epsilon:.0e}",
    showarrow=False,
    font=dict(color="white", size=12)
)

fig.update_layout(
    template=TEMPLATE,
    title="Divergencia exponencial en el sistema de Lorenz",
    xaxis_title="Tiempo",
    yaxis_title="log₁₀ |δ(t)|",
    width=800, height=400
)
fig.show()


### 3.3 Superficies 3D — Potencial gravitacional

In [ ]:
# Potencial gravitacional de dos masas puntuales en el plano z=0
N_grid = 120
xg = np.linspace(-3, 3, N_grid)
yg = np.linspace(-3, 3, N_grid)
Xg, Yg = np.meshgrid(xg, yg)

# Dos masas en (-1, 0) y (1, 0)
r1 = np.sqrt((Xg + 1)**2 + Yg**2) + 0.1
r2 = np.sqrt((Xg - 1)**2 + Yg**2) + 0.1

Phi = -1.0/r1 - 1.0/r2
Phi_clip = np.clip(Phi, -5, 0)  # recortar singularidades

fig = go.Figure(go.Surface(
    x=Xg, y=Yg, z=Phi_clip,
    colorscale="RdPu_r",
    opacity=0.9,
    contours=dict(
        z=dict(show=True, usecolormap=True, highlightcolor="white", project_z=True)
    )
))

fig.update_layout(
    template=TEMPLATE,
    title="Potencial gravitacional Φ(x,y) de dos masas puntuales",
    scene=dict(
        xaxis_title="x (UA)",
        yaxis_title="y (UA)",
        zaxis_title="Φ (u.a.)",
        bgcolor="rgb(5,5,20)"
    ),
    width=800, height=600
)
fig.show()


### 3.4 Problema de 3 Cuerpos en 3D

El problema de tres cuerpos no tiene solución analítica general.
Numéricamente integramos las ecuaciones de Newton para tres masas bajo interacción mutua:

$$m_i \ddot{\vec{r}}_i = G \sum_{j \neq i} \frac{m_j (\vec{r}_j - \vec{r}_i)}{|\vec{r}_j - \vec{r}_i|^3}$$

Usaremos la **solución en figura de ocho** de Chenciner & Montgomery (2000), 
una órbita periódica estable de tres cuerpos iguales.


In [ ]:
# Figura de ocho: condiciones iniciales de Chenciner & Montgomery
# m = 1 (unidades adimensionales), G = 1
G3 = 1.0

# Posiciones y velocidades iniciales (Chenciner & Montgomery, 2000)
# fuente: Šuvakov & Dmitrašinović (2013), arXiv:1303.0181
m3 = np.array([1.0, 1.0, 1.0])

r0 = np.array([
    [-0.97000436,  0.24308753, 0.0],
    [ 0.97000436, -0.24308753, 0.0],
    [ 0.0,         0.0,        0.0]
])

v3_cm = np.array([ 0.93240737/2,  0.86473146/2, 0.0])
v0 = np.array([
    [ 0.93240737/2,  0.86473146/2, 0.0],
    [ 0.93240737/2,  0.86473146/2, 0.0],
    [-0.93240737,   -0.86473146,   0.0]
])

def three_body(t, y):
    # y: [r1x,r1y,r1z, r2x,r2y,r2z, r3x,r3y,r3z,
    #      v1x,v1y,v1z, v2x,v2y,v2z, v3x,v3y,v3z]
    pos = y[:9].reshape(3, 3)
    vel = y[9:].reshape(3, 3)
    acc = np.zeros((3, 3))
    for i in range(3):
        for j in range(3):
            if i != j:
                d = pos[j] - pos[i]
                r = np.linalg.norm(d)
                acc[i] += G3 * m3[j] * d / r**3
    return np.concatenate([vel.ravel(), acc.ravel()])

y0_3b = np.concatenate([r0.ravel(), v0.ravel()])

t_span_3b = (0, 6.3259)   # período aproximado
t_eval_3b = np.linspace(*t_span_3b, 5000)

sol_3b = solve_ivp(three_body, t_span_3b, y0_3b, t_eval=t_eval_3b,
                   method="DOP853", rtol=1e-11, atol=1e-13)

pos_sol = sol_3b.y[:9].reshape(3, 3, -1)  # (3 cuerpos, 3 coords, N_t)

colores3 = ["deepskyblue", "tomato", "limegreen"]
nombres3 = ["Cuerpo 1", "Cuerpo 2", "Cuerpo 3"]

fig = go.Figure()

for i in range(3):
    fig.add_trace(go.Scatter3d(
        x=pos_sol[i, 0],
        y=pos_sol[i, 1],
        z=pos_sol[i, 2],
        mode="lines",
        line=dict(color=colores3[i], width=3),
        name=nombres3[i]
    ))
    # Posición final
    fig.add_trace(go.Scatter3d(
        x=[pos_sol[i, 0, -1]],
        y=[pos_sol[i, 1, -1]],
        z=[pos_sol[i, 2, -1]],
        mode="markers",
        marker=dict(color=colores3[i], size=8),
        showlegend=False
    ))

fig.update_layout(
    template=TEMPLATE,
    title="Solución en figura de ocho — problema de 3 cuerpos",
    scene=dict(
        xaxis_title="x",
        yaxis_title="y",
        zaxis_title="z",
        bgcolor="rgb(5,5,20)"
    ),
    width=750, height=620
)
fig.show()


### 3.5 Mapa de calor 3D — distribución de energía en una cuerda vibrante

In [ ]:
# Solución analítica de la cuerda vibrante (ecuación de ondas)
# u(x,t) = Σ Aₙ sin(nπx/L) cos(nπct/L)

L_cuerda = 1.0
c_cuerda = 1.0
N_modos = 10
N_x = 200
N_t = 200

x_c = np.linspace(0, L_cuerda, N_x)
t_c = np.linspace(0, 2, N_t)
Xc, Tc = np.meshgrid(x_c, t_c)

# Condición inicial: pluck en el centro
u_xt = np.zeros_like(Xc)
for n in range(1, N_modos + 1):
    An = (8 * np.sin(n * np.pi / 2)) / (n * np.pi)**2
    kn = n * np.pi / L_cuerda
    wn = c_cuerda * kn
    u_xt += An * np.sin(kn * Xc) * np.cos(wn * Tc)

fig = go.Figure(go.Surface(
    x=x_c, y=t_c, z=u_xt,
    colorscale="RdBu_r",
    cmin=-1, cmax=1,
    showscale=True
))

fig.update_layout(
    template=TEMPLATE,
    title="Cuerda vibrante: u(x, t)",
    scene=dict(
        xaxis_title="Posición x (m)",
        yaxis_title="Tiempo t (s)",
        zaxis_title="Desplazamiento u",
        bgcolor="rgb(5,5,20)",
        camera=dict(eye=dict(x=1.8, y=-1.5, z=1.2))
    ),
    width=800, height=580
)
fig.show()


---
## Sección 4 — Animaciones con Plotly

Las animaciones en Plotly se construyen con **frames**: cada frame define el estado de la
figura en un instante de tiempo. El widget de reproducción (`play/pause`) es generado
automáticamente.

```python
fig = go.Figure(
    data  = [trace_inicial],
    frames= [go.Frame(data=[trace_t]) for trace_t in ...],
    layout= go.Layout(
        updatemenus=[dict(type="buttons",
                         buttons=[dict(label="▶", method="animate", ...)])]
    )
)
```


### 4.1 Animación del péndulo simple

In [ ]:
# Integramos el péndulo una vez más, resolución alta
t_anim = np.linspace(0, 10, 500)
sol_pend_anim = solve_ivp(pendulo, (0, 10), [np.radians(60), 0],
                          t_eval=t_anim, rtol=1e-10)

theta_arr = sol_pend_anim.y[0]
L_pend = 1.0

# Coordenadas de la masa
xp = L_pend * np.sin(theta_arr)
yp = -L_pend * np.cos(theta_arr)

# Frames
frames = []
for k in range(len(t_anim)):
    frames.append(go.Frame(
        data=[
            # Cuerda
            go.Scatter(x=[0, xp[k]], y=[0, yp[k]],
                       mode="lines",
                       line=dict(color="white", width=3),
                       showlegend=False),
            # Masa
            go.Scatter(x=[xp[k]], y=[yp[k]],
                       mode="markers",
                       marker=dict(color="cyan", size=18),
                       showlegend=False),
            # Trayectoria
            go.Scatter(x=xp[:k+1], y=yp[:k+1],
                       mode="lines",
                       line=dict(color="cyan", width=1, dash="dot"),
                       showlegend=False)
        ],
        name=str(k)
    ))

fig_pend = go.Figure(
    data=[
        go.Scatter(x=[0, xp[0]], y=[0, yp[0]],
                   mode="lines", line=dict(color="white", width=3)),
        go.Scatter(x=[xp[0]], y=[yp[0]],
                   mode="markers", marker=dict(color="cyan", size=18)),
        go.Scatter(x=[], y=[], mode="lines",
                   line=dict(color="cyan", width=1, dash="dot"))
    ],
    frames=frames
)

fig_pend.update_layout(
    template=TEMPLATE,
    title="Animación del péndulo simple (θ₀ = 60°)",
    xaxis=dict(range=[-1.3, 1.3], title="x (m)", constrain="domain"),
    yaxis=dict(range=[-1.3, 0.3], title="y (m)", scaleanchor="x"),
    width=600, height=600,
    updatemenus=[dict(
        type="buttons",
        showactive=False,
        y=1.05, x=0.5, xanchor="center",
        buttons=[
            dict(label="▶ Play",
                 method="animate",
                 args=[None, dict(frame=dict(duration=20, redraw=True),
                                 fromcurrent=True, mode="immediate")]),
            dict(label="⏸ Pause",
                 method="animate",
                 args=[[None], dict(frame=dict(duration=0, redraw=False),
                                    mode="immediate")])
        ]
    )],
    sliders=[dict(
        steps=[dict(args=[[f.name], dict(frame=dict(duration=0, redraw=True),
                                         mode="immediate")],
                    method="animate",
                    label="") for f in frames],
        x=0.1, y=0, len=0.8
    )]
)
fig_pend.show()


### 4.2 Animación de órbita de dos cuerpos (Kepler)

In [ ]:
# Órbita de Marte durante 2 años
t_span_k2 = (0, 2)
t_eval_k2 = np.linspace(*t_span_k2, 400)

sol_k2 = solve_ivp(kepler_2d, t_span_k2, y0_marte, t_eval=t_eval_k2, rtol=1e-10)

xm, ym = sol_k2.y[0], sol_k2.y[1]

frames_k = []
for k in range(1, len(t_eval_k2)):
    frames_k.append(go.Frame(
        data=[
            go.Scatter(x=[0], y=[0],
                       mode="markers",
                       marker=dict(color="yellow", size=22, symbol="circle"),
                       showlegend=False),
            go.Scatter(x=xm[:k+1], y=ym[:k+1],
                       mode="lines",
                       line=dict(color="tomato", width=1.5),
                       showlegend=False),
            go.Scatter(x=[xm[k]], y=[ym[k]],
                       mode="markers",
                       marker=dict(color="tomato", size=14),
                       showlegend=False)
        ],
        name=str(k)
    ))

fig_kep = go.Figure(
    data=[
        go.Scatter(x=[0], y=[0], mode="markers",
                   marker=dict(color="yellow", size=22), name="Sol"),
        go.Scatter(x=[xm[0]], y=[ym[0]], mode="lines",
                   line=dict(color="tomato"), name="Órbita"),
        go.Scatter(x=[xm[0]], y=[ym[0]], mode="markers",
                   marker=dict(color="tomato", size=14), name="Marte")
    ],
    frames=frames_k
)

fig_kep.update_layout(
    template=TEMPLATE,
    title="Órbita de Marte — animación",
    xaxis=dict(range=[-2, 2], title="x (UA)", constrain="domain"),
    yaxis=dict(range=[-2, 2], title="y (UA)", scaleanchor="x"),
    width=620, height=620,
    updatemenus=[dict(
        type="buttons", showactive=False,
        y=1.07, x=0.5, xanchor="center",
        buttons=[
            dict(label="▶ Play", method="animate",
                 args=[None, dict(frame=dict(duration=30, redraw=True),
                                 fromcurrent=True)]),
            dict(label="⏸ Pause", method="animate",
                 args=[[None], dict(frame=dict(duration=0, redraw=False),
                                    mode="immediate")])
        ]
    )]
)
fig_kep.show()


### 4.3 Animación del problema de tres cuerpos

In [ ]:
# Reutilizamos la solución de figura de ocho
# Submuestreo para la animación
step_anim = 5
t_frames = list(range(0, pos_sol.shape[2], step_anim))
N_frames = len(t_frames)

frames_3b = []
for k in t_frames:
    traces_frame = []
    for i in range(3):
        # Trayectoria hasta el instante k
        start = max(0, k - 200)
        traces_frame.append(go.Scatter(
            x=pos_sol[i, 0, start:k+1],
            y=pos_sol[i, 1, start:k+1],
            mode="lines",
            line=dict(color=colores3[i], width=1.5),
            showlegend=False,
            opacity=0.6
        ))
        # Posición actual
        traces_frame.append(go.Scatter(
            x=[pos_sol[i, 0, k]],
            y=[pos_sol[i, 1, k]],
            mode="markers",
            marker=dict(color=colores3[i], size=14),
            showlegend=False
        ))
    frames_3b.append(go.Frame(data=traces_frame, name=str(k)))

# Traces iniciales
init_traces = []
for i in range(3):
    init_traces.append(go.Scatter(x=[], y=[], mode="lines",
                                  line=dict(color=colores3[i], width=1.5),
                                  name=nombres3[i]))
    init_traces.append(go.Scatter(x=[pos_sol[i, 0, 0]],
                                  y=[pos_sol[i, 1, 0]],
                                  mode="markers",
                                  marker=dict(color=colores3[i], size=14),
                                  showlegend=False))

fig_3b = go.Figure(data=init_traces, frames=frames_3b)

fig_3b.update_layout(
    template=TEMPLATE,
    title="Problema de 3 Cuerpos — figura de ocho (animación)",
    xaxis=dict(range=[-1.3, 1.3], title="x"),
    yaxis=dict(range=[-0.6, 0.6], title="y", scaleanchor="x"),
    width=750, height=450,
    updatemenus=[dict(
        type="buttons", showactive=False,
        y=1.07, x=0.5, xanchor="center",
        buttons=[
            dict(label="▶ Play", method="animate",
                 args=[None, dict(frame=dict(duration=20, redraw=True),
                                 fromcurrent=True)]),
            dict(label="⏸ Pause", method="animate",
                 args=[[None], dict(frame=dict(duration=0, redraw=False),
                                    mode="immediate")])
        ]
    )]
)
fig_3b.show()


### 4.4 Animación 3D del atractor de Lorenz

In [ ]:
# Submuestreo del atractor de Lorenz para animar
step_lor = 100
idx_frames = list(range(0, len(t_eval_lor), step_lor))

frames_lor = []
for k in idx_frames:
    tail = max(0, k - 3000)
    frames_lor.append(go.Frame(
        data=[go.Scatter3d(
            x=x_l[tail:k+1],
            y=y_c_l[tail:k+1],
            z=z_l[tail:k+1],
            mode="lines",
            line=dict(
                color=t_eval_lor[tail:k+1],
                colorscale="Plasma",
                width=2
            )
        )],
        name=str(k)
    ))

fig_lor_anim = go.Figure(
    data=[go.Scatter3d(x=[x_l[0]], y=[y_c_l[0]], z=[z_l[0]],
                       mode="lines",
                       line=dict(color="purple", width=2))],
    frames=frames_lor
)

fig_lor_anim.update_layout(
    template=TEMPLATE,
    title="Atractor de Lorenz — animación 3D",
    scene=dict(
        xaxis=dict(range=[-25, 25]),
        yaxis=dict(range=[-35, 35]),
        zaxis=dict(range=[0, 55]),
        xaxis_title="x",
        yaxis_title="y",
        zaxis_title="z",
        bgcolor="rgb(5,5,20)"
    ),
    width=800, height=600,
    updatemenus=[dict(
        type="buttons", showactive=False,
        y=1.05, x=0.5, xanchor="center",
        buttons=[
            dict(label="▶ Play", method="animate",
                 args=[None, dict(frame=dict(duration=30, redraw=True),
                                 fromcurrent=True)]),
            dict(label="⏸ Pause", method="animate",
                 args=[[None], dict(frame=dict(duration=0, redraw=False),
                                    mode="immediate")])
        ]
    )]
)
fig_lor_anim.show()


### 4.5 Bonus: Animación del Sistema Solar — múltiples órbitas

In [ ]:
# Datos de planetas del sistema solar interior + Júpiter
planetas_data = {
    "Mercurio": dict(a=0.387, e=0.205, color="gold",       size=6,  T=0.241),
    "Venus"   : dict(a=0.723, e=0.007, color="wheat",      size=9,  T=0.615),
    "Tierra"  : dict(a=1.000, e=0.017, color="deepskyblue",size=9,  T=1.000),
    "Marte"   : dict(a=1.524, e=0.093, color="tomato",     size=7,  T=1.881),
    "Júpiter" : dict(a=5.203, e=0.049, color="orange",     size=18, T=11.86),
}

def v_perihelio(a, e):
    return np.sqrt(G_au * (1 + e) / (a * (1 - e)))

t_sys = np.linspace(0, 12, 800)   # 12 años

orbitas = {}
for nombre, p in planetas_data.items():
    y0_p = [p["a"] * (1 - p["e"]), 0, 0, v_perihelio(p["a"], p["e"])]
    sol_p = solve_ivp(kepler_2d, (0, 12), y0_p, t_eval=t_sys, rtol=1e-10)
    orbitas[nombre] = (sol_p.y[0], sol_p.y[1])

# Construir frames
frames_ss = []
for k in range(len(t_sys)):
    traces_f = [go.Scatter(x=[0], y=[0], mode="markers",
                           marker=dict(color="yellow", size=24),
                           showlegend=False)]
    for nombre, p in planetas_data.items():
        xorb, yorb = orbitas[nombre]
        # Trayectoria completa (órbita)
        traces_f.append(go.Scatter(
            x=xorb, y=yorb,
            mode="lines",
            line=dict(color=p["color"], width=1),
            opacity=0.3,
            showlegend=False
        ))
        # Planeta en el instante k
        traces_f.append(go.Scatter(
            x=[xorb[k]], y=[yorb[k]],
            mode="markers+text",
            marker=dict(color=p["color"], size=p["size"]),
            text=[nombre], textposition="top center",
            textfont=dict(size=10, color=p["color"]),
            showlegend=False
        ))
    frames_ss.append(go.Frame(data=traces_f, name=str(k)))

# Traces iniciales
init_ss = [go.Scatter(x=[0], y=[0], mode="markers",
                      marker=dict(color="yellow", size=24), name="Sol")]
for nombre, p in planetas_data.items():
    xorb, yorb = orbitas[nombre]
    init_ss.append(go.Scatter(x=xorb, y=yorb, mode="lines",
                               line=dict(color=p["color"], width=1),
                               opacity=0.3, showlegend=False))
    init_ss.append(go.Scatter(x=[xorb[0]], y=[yorb[0]],
                               mode="markers+text",
                               marker=dict(color=p["color"], size=p["size"]),
                               text=[nombre], textposition="top center",
                               textfont=dict(size=10, color=p["color"]),
                               name=nombre))

fig_ss = go.Figure(data=init_ss, frames=frames_ss)

fig_ss.update_layout(
    template=TEMPLATE,
    title="Sistema Solar — animación (12 años)",
    xaxis=dict(range=[-6, 6], title="x (UA)", constrain="domain"),
    yaxis=dict(range=[-6, 6], title="y (UA)", scaleanchor="x"),
    width=700, height=680,
    updatemenus=[dict(
        type="buttons", showactive=False,
        y=1.06, x=0.5, xanchor="center",
        buttons=[
            dict(label="▶ Play", method="animate",
                 args=[None, dict(frame=dict(duration=25, redraw=True),
                                 fromcurrent=True)]),
            dict(label="⏸ Pause", method="animate",
                 args=[[None], dict(frame=dict(duration=0, redraw=False),
                                    mode="immediate")])
        ]
    )]
)
fig_ss.show()


---
## Sección 5 — Exportar figuras y Buenas Prácticas

### 5.1 Guardar como HTML interactivo
```python
fig.write_html("mi_figura.html")
```

### 5.2 Guardar como imagen estática (requiere `kaleido`)
```python
fig.write_image("mi_figura.png", width=1200, height=800, scale=2)
fig.write_image("mi_figura.pdf")
fig.write_image("mi_figura.svg")
```

### 5.3 Incrustar en un notebook sin conexión
```python
import plotly.io as pio
pio.renderers.default = "notebook"   # o "jupyterlab", "vscode"
```

### 5.4 Resumen de buenas prácticas

| Recomendación | Por qué |
|---|---|
| Usar `template=TEMPLATE` globalmente | Consistencia visual |
| `yaxis=dict(scaleanchor="x")` en órbitas | Mantener proporciones físicas |
| `hovertemplate` personalizado | Información física en el tooltip |
| Submuestrear con `step` en animaciones | Reducir tiempo de render |
| `rtol=1e-9`, `method="DOP853"` | Precisión en EDOs caóticas |
| Separar `solve_ivp` de la visualización | Código más limpio y reutilizable |


---
## Resumen

En este cuaderno aprendiste a:

1. **Plotly 2D**: crear gráficas de líneas, dispersión, subplots, espacios de fase y campos vectoriales.
2. **Sistemas de EDOs**: integrar y visualizar el péndulo, oscilador amortiguado forzado, Lotka-Volterra y órbitas de Kepler.
3. **Plotly 3D**: construir curvas, superficies y el atractor de Lorenz en 3D interactivo.
4. **Animaciones**: usar frames y botones Play/Pause para animar péndulos, órbitas, el problema de 3 cuerpos y el atractor de Lorenz.

### Recursos adicionales

- [Documentación oficial Plotly Python](https://plotly.com/python/)
- [Plotly Express — API de alto nivel](https://plotly.com/python/plotly-express/)
- [Scipy `solve_ivp`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.integrate.solve_ivp.html)
- [Šuvakov & Dmitrašinović (2013) — órbitas periódicas del problema de 3 cuerpos](https://arxiv.org/abs/1303.0181)
- [Chenciner & Montgomery (2000) — figura de ocho](https://arxiv.org/abs/math/0011268)
